<a href="https://colab.research.google.com/github/RatanakamonS/Stock_Price/blob/main/Optimization_Portfolio_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, linprog

In [2]:
# =========================================================
# CONFIG
# =========================================================
BETA = 0.95
TAIL = 1 - BETA
TOL  = 1e-6

URL_ASSETS = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/3yrs_clean_sp500_adjusted_close_prices.csv"
URL_GSPC   = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/gspc_ohlcv.csv"
URL_SHARES = "https://raw.githubusercontent.com/RatanakamonS/Stock_Price/main/sp500_shares_outstanding_proxy.csv"

In [3]:
# =========================================================
# IO HELPERS
# =========================================================
def load_asset_returns(url_assets: str) -> pd.DataFrame:
    prices = pd.read_csv(url_assets, index_col=0)
    prices.index = pd.to_datetime(prices.index, errors="coerce", dayfirst=True)
    prices = prices[~prices.index.isna()].sort_index()
    prices = prices.apply(pd.to_numeric, errors="raise")
    return prices.pct_change().dropna()

def load_gspc_adj_close(url_gspc: str) -> pd.Series:
    gspc = pd.read_csv(url_gspc)
    gspc = gspc.rename(columns={gspc.columns[0]: "Date"})
    gspc["Date"] = pd.to_datetime(gspc["Date"], errors="coerce")
    gspc = gspc.dropna(subset=["Date"]).set_index("Date").sort_index()

    if "Adj Close" not in gspc.columns:
        raise KeyError("Market CSV missing column: 'Adj Close'")

    gspc["Adj Close"] = pd.to_numeric(gspc["Adj Close"], errors="coerce")
    gspc = gspc.dropna(subset=["Adj Close"])
    return gspc["Adj Close"]

def export_market_weights(url_shares: str, url_prices: str, out_csv: str = "sp500_market_portfolio_weights.csv"):
    shares_df = pd.read_csv(url_shares).rename(
        columns={"symbol": "Ticker", "sharesOutstanding_proxy": "Shares"}
    )
    shares_df = shares_df.dropna(subset=["Shares"])

    prices = pd.read_csv(url_prices, index_col=0)
    prices.index = pd.to_datetime(prices.index, errors="coerce")
    prices = prices.sort_index()

    target_date = prices.index.max()
    print("Using price date:", target_date.date())

    prices_t = prices.loc[target_date].reset_index()
    prices_t.columns = ["Ticker", "Price"]

    mkt_df = shares_df.merge(prices_t, on="Ticker", how="inner")
    mkt_df["MarketCap"] = mkt_df["Shares"] * mkt_df["Price"]
    mkt_df["Weight"] = mkt_df["MarketCap"] / mkt_df["MarketCap"].sum()

    mkt_df = mkt_df.sort_values("Weight", ascending=False)
    print(mkt_df.head(10))
    print("Sum of weights:", float(mkt_df["Weight"].sum()))

    mkt_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

def save_market_return_csv(gspc_adj_close: pd.Series, out_csv: str = "gspc_market_return.csv"):
    gspc_ret = gspc_adj_close.pct_change()
    out = pd.DataFrame({"Adj Close": gspc_adj_close, "mkt_ret": gspc_ret}).dropna()
    out.to_csv(out_csv, index=True)
    print(f"Saved: {out_csv}")

In [4]:
# =========================================================
# MATH HELPERS
# =========================================================
def portfolio_variance(w: np.ndarray, Sigma: np.ndarray) -> float:
    w = np.asarray(w, dtype=float).reshape(-1)
    return float(w @ Sigma @ w)

def portfolio_returns(R: np.ndarray, w: np.ndarray) -> np.ndarray:
    return (R @ w).astype(float)

def empirical_var_cvar_from_port_returns(port_ret: np.ndarray, beta: float):
    """
    Historical Simulation (Empirical) VaR/CVaR of LOSS = -Return
      L_t = -R_{p,t}
      VaR_beta = sample quantile at beta
      CVaR_beta = mean(L_t | L_t >= VaR_beta)
    """
    L = -np.asarray(port_ret, dtype=float).reshape(-1)
    if L.size == 0:
        return np.nan, np.nan

    VaR = float(np.quantile(L, beta, method="linear")) if hasattr(np, "quantile") else float(pd.Series(L).quantile(beta))
    tail_losses = L[L >= VaR]
    CVaR = float(np.mean(tail_losses)) if tail_losses.size > 0 else VaR
    return VaR, CVaR

def solver_report(name, success, message, w, mu_assets, mu_market, long_only=False, tol=1e-6):
    print("\n" + "=" * 80)
    print(f"Solver Report: {name}")
    print(f"success : {success}")
    print(f"message : {message}")

    if (not success) or (w is None):
        print("=" * 80)
        return

    sum_w = float(np.sum(w))
    ret_w = float(mu_assets @ w)
    min_w = float(np.min(w))

    budget_ok = abs(sum_w - 1.0) <= tol
    return_ok = abs(ret_w - float(mu_market)) <= tol
    long_ok   = (min_w + tol) >= 0.0 if long_only else True

    print("-" * 80)
    print(f"sum(w)          = {sum_w:.10f} | budget_ok = {budget_ok}")
    print(f"mu_assets @ w   = {ret_w:.10f}")
    print(f"mu_market       = {float(mu_market):.10f}")
    print(f"abs(diff)       = {abs(ret_w - float(mu_market)):.10e} | return_eq_ok = {return_ok}")
    print(f"min(w)          = {min_w:.10f} | long_only_ok = {long_ok}")
    print(f"FEASIBLE(manual)= {bool(budget_ok and return_ok and long_ok)}")
    print("=" * 80)

In [5]:
# =========================================================
# OPTIMIZATION: MV (SLSQP)
# =========================================================
def solve_mv_equal_return(mu_assets: np.ndarray, Sigma: np.ndarray, mu_market: float, long_only: bool):
    """
    Mean–Variance:
      min  w' Σ w
      s.t. sum(w) = 1
           mu'w   = mu_market
           (optional) w_i >= 0
    """
    N = len(mu_assets)

    def mv_obj(w):
        return float(w @ Sigma @ w)

    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "eq", "fun": lambda w: (mu_assets @ w) - float(mu_market)},
    ]

    w0 = np.ones(N) / N
    bounds = [(0.0, 1.0)] * N if long_only else [(None, None)] * N

    res = minimize(
        mv_obj, w0,
        constraints=cons,
        bounds=bounds,
        method="SLSQP",
        options={"maxiter": 20000, "ftol": 1e-6}
    )
    return res

In [6]:
# =========================================================
# OPTIMIZATION: CVaR LP (Rockafellar–Uryasev)
# Variables: x = [w(1..N), u(1..T), alpha]
# minimize: alpha + (1/(T*tail)) * sum(u_t)
# s.t.      u_t >= -r_t'w - alpha
#           u_t >= 0
#           sum(w) = 1
#           mu'w  = mu_market
#           (optional) w_i >= 0
# =========================================================
def build_cvar_lp_equal_return(R, mu_assets, mu_market, beta, long_only=False):
    T, N = R.shape
    tail = 1 - beta

    c = np.zeros(N + T + 1)
    c[N:N+T] = 1.0 / (T * tail)   # u
    c[-1] = 1.0                   # alpha

    # u_t >= -r_t'w - alpha  <=>  -r_t'w - u_t - alpha <= 0
    A_ub = np.zeros((T, N + T + 1))
    b_ub = np.zeros(T)
    for t in range(T):
        A_ub[t, :N]    = -R[t, :]
        A_ub[t, N + t] = -1.0
        A_ub[t, -1]    = -1.0

    A_eq = np.zeros((2, N + T + 1))
    b_eq = np.zeros(2)

    A_eq[0, :N] = 1.0
    b_eq[0] = 1.0

    A_eq[1, :N] = mu_assets
    b_eq[1] = float(mu_market)

    w_bounds = [(0.0, None)] * N if long_only else [(None, None)] * N
    u_bounds = [(0.0, None)] * T
    alpha_bounds = [(None, None)]
    bounds = w_bounds + u_bounds + alpha_bounds

    return c, A_ub, b_ub, A_eq, b_eq, bounds

def solve_cvar_equal_return(R, mu_assets, mu_market, beta, long_only: bool):
    T, N = R.shape
    c, A_ub, b_ub, A_eq, b_eq, bounds = build_cvar_lp_equal_return(
        R=R, mu_assets=mu_assets, mu_market=mu_market, beta=beta, long_only=long_only
    )
    res = linprog(
        c, A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=bounds, method="highs"
    )
    if not res.success:
        return res, None, None, None

    x = res.x
    w = x[:N]
    u = x[N:N+T]
    alpha = x[-1]  # kept internally but not exported in summary per request
    return res, w, u, alpha

In [7]:
# =========================================================
# MAIN
# =========================================================
def main():
    # 0) Export market-cap weights
    export_market_weights(URL_SHARES, URL_ASSETS, out_csv="sp500_market_portfolio_weights.csv")

    # 1) Load asset returns
    print("\nLoading asset prices from GitHub...")
    asset_ret = load_asset_returns(URL_ASSETS)
    tickers = asset_ret.columns.tolist()

    # 2) Load market (^GSPC) adj close + compute returns + align dates
    print("Loading market index (^GSPC) from GitHub CSV...")
    gspc_adj = load_gspc_adj_close(URL_GSPC)
    gspc_ret = gspc_adj.pct_change().dropna()

    common_dates = asset_ret.index.intersection(gspc_ret.index)
    asset_ret = asset_ret.loc[common_dates]
    gspc_adj  = gspc_adj.loc[common_dates]
    gspc_ret  = gspc_ret.loc[common_dates]

    T, N = asset_ret.shape
    mu_market = float(gspc_ret.mean())

    print(f"Aligned data: N={N}, T={T}")
    print(f"mu_market (^GSPC mean return) = {mu_market:.10f}")

    # save market return csv (Adj Close + return)
    save_market_return_csv(gspc_adj_close=gspc_adj, out_csv="gspc_market_return.csv")

    # 3) Moments
    mu_assets = asset_ret.mean().values
    Sigma = asset_ret.cov().values
    R = asset_ret.values  # T x N

    # =====================================================
    # 4) Run 4 models
    # =====================================================

    # --- Model 1: MV Allow Short
    res_mv_allow = solve_mv_equal_return(mu_assets, Sigma, mu_market, long_only=False)
    print("\n" + "="*80)
    print("Solver Report: Mean–Variance (Allow Short)")
    print(f"success : {res_mv_allow.success}")
    print(f"message : {res_mv_allow.message}")
    print("="*80)
    if not res_mv_allow.success:
        raise RuntimeError("MV (Allow Short) optimization failed. Check feasibility: mu_market may be unreachable.")

    w_mv_allow = res_mv_allow.x
    solver_report("MV (Allow Short)", res_mv_allow.success, res_mv_allow.message,
                  w_mv_allow, mu_assets, mu_market, long_only=False, tol=TOL)

    Rp_mv_allow = portfolio_returns(R, w_mv_allow)
    mu_p_mv_allow = float(np.mean(Rp_mv_allow))
    var_mv_allow = portfolio_variance(w_mv_allow, Sigma)
    VaR_mv_allow_loss_emp, CVaR_mv_allow_loss_emp = empirical_var_cvar_from_port_returns(Rp_mv_allow, BETA)

    # --- Model 3: MV Long-only
    res_mv_long = solve_mv_equal_return(mu_assets, Sigma, mu_market, long_only=True)
    print("\n" + "="*80)
    print("Solver Report: Mean–Variance (Long-only)")
    print(f"success : {res_mv_long.success}")
    print(f"message : {res_mv_long.message}")
    print("="*80)
    if not res_mv_long.success:
        raise RuntimeError("MV (Long-only) optimization failed. Check feasibility under long-only.")

    w_mv_long = res_mv_long.x
    solver_report("MV (Long-only)", res_mv_long.success, res_mv_long.message,
                  w_mv_long, mu_assets, mu_market, long_only=True, tol=TOL)

    Rp_mv_long = portfolio_returns(R, w_mv_long)
    mu_p_mv_long = float(np.mean(Rp_mv_long))
    var_mv_long = portfolio_variance(w_mv_long, Sigma)
    VaR_mv_long_loss_emp, CVaR_mv_long_loss_emp = empirical_var_cvar_from_port_returns(Rp_mv_long, BETA)

    # --- Model 2: CVaR Allow Short
    res_cvar_allow, w_cvar_allow, u_allow, _alpha_allow = solve_cvar_equal_return(
        R=R, mu_assets=mu_assets, mu_market=mu_market, beta=BETA, long_only=False
    )
    solver_report("CVaR (Allow Short)", res_cvar_allow.success, res_cvar_allow.message,
                  w_cvar_allow, mu_assets, mu_market, long_only=False, tol=TOL)

    if res_cvar_allow.success:
        Rp_cvar_allow = portfolio_returns(R, w_cvar_allow)
        mu_p_cvar_allow = float(np.mean(Rp_cvar_allow))
        var_cvar_allow  = portfolio_variance(w_cvar_allow, Sigma)

        # Empirical VaR/CVaR only (no opt alpha/obj in summary)
        VaR_allow_loss_emp, CVaR_allow_loss_emp = empirical_var_cvar_from_port_returns(Rp_cvar_allow, BETA)

        # objective value ("Objective" ของโมเดล CVaR)
        obj_cvar_allow = float(res_cvar_allow.fun)
    else:
        mu_p_cvar_allow = np.nan
        var_cvar_allow = np.nan
        VaR_allow_loss_emp = np.nan
        CVaR_allow_loss_emp = np.nan
        obj_cvar_allow = np.nan

    # --- Model 4: CVaR Long-only
    res_cvar_long, w_cvar_long, u_long, _alpha_long = solve_cvar_equal_return(
        R=R, mu_assets=mu_assets, mu_market=mu_market, beta=BETA, long_only=True
    )
    solver_report("CVaR (Long-only)", res_cvar_long.success, res_cvar_long.message,
                  w_cvar_long, mu_assets, mu_market, long_only=True, tol=TOL)

    if res_cvar_long.success:
        Rp_cvar_long = portfolio_returns(R, w_cvar_long)
        mu_p_cvar_long = float(np.mean(Rp_cvar_long))
        var_cvar_long  = portfolio_variance(w_cvar_long, Sigma)

        # Empirical VaR/CVaR only
        VaR_long_loss_emp, CVaR_long_loss_emp = empirical_var_cvar_from_port_returns(Rp_cvar_long, BETA)

        obj_cvar_long = float(res_cvar_long.fun)
    else:
        mu_p_cvar_long = np.nan
        var_cvar_long = np.nan
        VaR_long_loss_emp = np.nan
        CVaR_long_loss_emp = np.nan
        obj_cvar_long = np.nan


    # =====================================================
    # 5) Export summary + weights
    # =====================================================
    summary = pd.DataFrame({
        "Model": [
            "Mean–Variance (Allow Short)",
            "CVaR (Allow Short)",
            "Mean–Variance (Long-only)",
            "CVaR (Long-only)",
        ],

        # mu ของพอร์ต (realized mean ของ Rp,t)
        "mu_port": [
            mu_p_mv_allow,
            mu_p_cvar_allow,
            mu_p_mv_long,
            mu_p_cvar_long,
        ],

        # objective ของแต่ละโมเดล
        "Objective": [
            var_mv_allow,       # MV minimize variance
            obj_cvar_allow,     # CVaR LP minimize CVaR loss (objective)
            var_mv_long,
            obj_cvar_long,
        ],

        "Variance": [
            var_mv_allow,
            var_cvar_allow,
            var_mv_long,
            var_cvar_long,
        ],

        # VaR/CVaR แบบ Empirical (Historical Simulation) สำหรับ “ทุกโมเดล”
        "VaR_0.95_Loss": [
            VaR_mv_allow_loss_emp,
            VaR_allow_loss_emp,
            VaR_mv_long_loss_emp,
            VaR_long_loss_emp,
        ],
        "CVaR_0.95_Loss": [
            CVaR_mv_allow_loss_emp,
            CVaR_allow_loss_emp,
            CVaR_mv_long_loss_emp,
            CVaR_long_loss_emp,
        ],

        "Solver_success": [
            bool(res_mv_allow.success),
            bool(res_cvar_allow.success),
            bool(res_mv_long.success),
            bool(res_cvar_long.success),
        ],
        "Solver_message": [
            str(res_mv_allow.message),
            str(res_cvar_allow.message),
            str(res_mv_long.message),
            str(res_cvar_long.message),
        ]
    })

    weights = pd.DataFrame({
        "Ticker": tickers,
        "w_MV_allow": w_mv_allow,
        "w_CVaR_allow": (w_cvar_allow if res_cvar_allow.success else np.full(N, np.nan)),
        "w_MV_long": w_mv_long,
        "w_CVaR_long": (w_cvar_long if res_cvar_long.success else np.full(N, np.nan)),
    })

    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(summary.to_string(index=False))

    print("\n" + "="*80)
    print("WEIGHTS (first 20 rows)")
    print("="*80)
    print(weights.head(20).to_string(index=False))

    summary.to_csv("summary_objectives.csv", index=False)
    weights.to_csv("weights_all_models.csv", index=False)
    print("\nSaved: summary_objectives.csv, weights_all_models.csv")

In [8]:
if __name__ == "__main__":
    main()

Using price date: 2025-12-09
    Ticker       Shares        Price     MarketCap    Weight
326   NFLX   4237323340  1188.439941  5.035804e+12  0.080559
340   NVDA  24305000000   177.820007  4.321915e+12  0.069139
310   MSFT   7432377655   509.899994  3.789769e+12  0.060626
38    AAPL  14697926000   234.070007  3.440344e+12  0.055036
22    AMZN  10690216011   228.149994  2.438973e+12  0.039017
71    AVGO   4741273799   359.254456  1.703324e+12  0.027249
304   META   2177889269   755.080383  1.644481e+12  0.026307
19   GOOGL   5818000000   240.800003  1.400974e+12  0.022412
433   TSLA   3325819167   395.940002  1.316825e+12  0.021066
20    GOOG   5407000000   241.380005  1.305142e+12  0.020879
Sum of weights: 1.0
Saved: sp500_market_portfolio_weights.csv

Loading asset prices from GitHub...
Loading market index (^GSPC) from GitHub CSV...
Aligned data: N=495, T=751
mu_market (^GSPC mean return) = 0.0008092511
Saved: gspc_market_return.csv

Solver Report: Mean–Variance (Allow Short)
success